In [1]:
import sys
import logging
from html2text import HTML2Text
from playwright.async_api import async_playwright
import nest_asyncio
import asyncio
from typing import List
from threading import Thread
import os
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, load_index_from_storage, SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.base.base_query_engine import BaseQueryEngine
from dotenv import load_dotenv
load_dotenv()
nest_asyncio.apply()

c:\Users\Abhinav\Desktop\LLM_portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import openai
print("openai version:", openai.__version__)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

openai version: 2.20.0


In [4]:
chat_llm = AzureOpenAI(engine = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
                       model = "gpt-4o",
                       api_key = os.getenv('AZURE_OPENAI_API_KEY'),
                       api_version = os.getenv('AZURE_OPENAI_API_VERSION'),
                                        azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT'))
                                        
embedding_llm = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = chat_llm
Settings.embed_model = embedding_llm

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolv

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary R

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 438.82it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 30

In [5]:
async def download_and_save_in_markdown(
                                url: str, 
                                dir_path: str,
                                browser_context
                                ) -> None:
    """Download the HTML content using Playwright and save as markdown."""
    
    if url.endswith("/"):
        url = url[:-1]

    filename = url.split("/")[-1] + ".md"
    file_path = os.path.join(dir_path, filename)
    if os.path.exists(file_path):
        return

    print(f"Downloading {url}")
    page = await browser_context.new_page()
    try:
        await page.goto(url,wait_until="networkidle",timeout=60000)
        content = await page.content()
        h = HTML2Text()
        h.ignore_links = False
        markdown_content = h.handle(content)
        with open(file_path,"w",encoding="utf-8") as f:
            f.write(markdown_content)
    except Exception as e:
        print(f"Failed {url}:{e}")
    finally:
        await page.close()
def run_async_in_thread(coro):
    result=None
    def wrapper():
        nonlocal result
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        result = loop.run_until_complete(coro)
        loop.close()
    thread = Thread()
    thread.start()
    thread.join()
    return result
    
async def download_logic(pages: List[str]) -> str:
    dir_path = "blogs"
    os.makedirs(dir_path, exist_ok=True)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        tasks = [download_and_save_in_markdown(page, dir_path, context) for page in pages]
        await asyncio.gather(*tasks)
        await browser.close()
    return dir_path


PAGES = [
    "https://quickstarts.snowflake.com/guide/data_engineering_pipelines_with_snowpark_python",
    "https://quickstarts.snowflake.com/guide/cloud_native_data_engineering_with_matillion_and_snowflake",
    "https://quickstarts.snowflake.com/guide/data_engineering_with_apache_airflow",
    "https://quickstarts.snowflake.com/guide/getting_started_with_dataengineering_ml_using_snowpark_python",
    "https://quickstarts.snowflake.com/guide/data_engineering_with_snowpark_python_and_dbt"
]

run_async_in_thread(download_logic(PAGES))

C:\Users\Abhinav\AppData\Local\Temp\ipykernel_58524\4256177041.py:65: RuntimeWarning: coroutine 'download_logic' was never awaited
  run_async_in_thread(download_logic(PAGES))


In [9]:
def build_index(
                data_dir: str, 
                knowledge_base_dir: str
                ) -> None:
    """Build the vector index from the markdown files in the directory."""

    print("Building vector index...")
    documents = SimpleDirectoryReader(data_dir).load_data()

    # index = TreeIndex.from_documents(documents, service_context=service_context)
    index = VectorStoreIndex.from_documents(
                                                    documents, 
                                                    show_progress=True
                                                    )
    index.storage_context.persist(persist_dir=knowledge_base_dir)
    print("Done.")

In [11]:
build_index('blogs/', 'kb/')

Building vector index...


Generating embeddings: 100%|██████████| 57/57 [00:12<00:00,  4.42it/s]

Done.


In [12]:
def load_index(knowledge_base_dir: str) -> BaseQueryEngine:
    """Load the vector index from the directory."""
    print("Loading vector index...")
    storage_context = StorageContext.from_defaults(persist_dir=knowledge_base_dir)
    index = load_index_from_storage(storage_context=storage_context)
    query_engine = index.as_query_engine()
    print("Done.")
    return query_engine

In [13]:
query_engine = load_index('kb/')

Loading vector index...
INFO:llama_index.core.indices.loading:Loading all indices.
Loading all indices.
Done.


In [14]:
def chat_inference(
                    query: str,
                    top_k: int = 5
                    ) -> None:
    """Run a query against the index and print the results."""
    results = query_engine.query(query)
    print(f"Query: {query}")
    print(f"Response: {results}")

In [15]:
chat_inference("what data engineers are focused primarily ?")

INFO:httpx:HTTP Request: POST https://dev-openai-service-03.openai.azure.com/openai/deployments/janbatch2-5503e7d5-7250-4c30-a4f9-72adefebd42e/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
HTTP Request: POST https://dev-openai-service-03.openai.azure.com/openai/deployments/janbatch2-5503e7d5-7250-4c30-a4f9-72adefebd42e/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
Query: what data engineers are focused primarily ?
Response: Data engineers are primarily focused on building and maintaining data pipelines that automate the transfer of data through various steps, transforming it into a usable format for specific types of analysis. This involves collecting, preparing, transforming, and delivering data as part of an ongoing practice.
